# Synthetic Data Generator

In [ ]:
import os
from dotenv import load_dotenv
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch

In [ ]:

MODEL_LLAMA = "meta-llama/Llama-3.2-1B-Instruct"
MODEL_LLAMA_KEY = "LLAMA"
MODEL_PHI = "microsoft/Phi-4-mini-instruct"
MODEL_PHI_KEY = "PHI4"
MODEL_DEEPSEEK = "deepseek-ai/DeepSeek-V3.1"
MODEL_DEEPSEEK_KEY = "DEEPSEEK"

models = {
    MODEL_LLAMA_KEY: MODEL_LLAMA,
    MODEL_PHI_KEY: MODEL_PHI,
    MODEL_DEEPSEEK_KEY: MODEL_DEEPSEEK
}

In [ ]:

load_dotenv(override=True)

In [ ]:
system_prompt = "You are a synthetic data generator. Your task is to generate Kaggle or Huggingface-type datasets that can be used in data science projects."

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

class ModelManager:
    def __init__(self):
        self._model = None
        self._model_key = None
        self._is_quantized = False

    def get_model(self, model_key, enable_quantization):
        cache_hit = (
          self._model is not None and
          self._model_key == model_key and
          self._is_quantized == enable_quantization
        )
        if not cache_hit:
            self._is_quantized = enable_quantization
            selected_model = models[model_key]
            if enable_quantization:
                self._model = AutoModelForCausalLM.from_pretrained(selected_model, device_map="auto", quantization_config=quant_config)
            else:
                self._model = AutoModelForCausalLM.from_pretrained(selected_model, device_map="auto")
            self._model_key = model_key
        return self._model

model_manager = ModelManager()

In [ ]:
def generate_synthetic_data(history, model_key, enable_quantization):
    system_message = [{"role": "system", "content": system_prompt}]
    history_messages =  [{"role": h["role"], "content": h["content"]} for h in history]
    messages = system_message + history_messages
    selected_model = models[model_key]

    tokenizer = AutoTokenizer.from_pretrained(selected_model)
    tokenizer.pad_token = tokenizer.eos_token
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True)["input_ids"].to("cuda")
    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
    streamer= TextStreamer(tokenizer)

    model = model_manager.get_model(model_key, enable_quantization)
    
    outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, streamer=streamer, max_new_tokens=2048)
    # only decode the newly generated tokens
    input_length = input_ids.shape[1]
    result = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    
    yield history + [{"role": "assistant", "content": result}]

In [ ]:
# Build Gradio interface

# This is a helper function to put the user message into the chatbot history before we call the chat function.
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

def reset_history():
    return []

with gr.Blocks() as ui:
    gr.Markdown("## Synthetic Data Generator with LLMs")
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
    with gr.Row():
         model_selector = gr.Dropdown(models.keys(), label="Select model", value=MODEL_LLAMA_KEY)
         model_selector.change(reset_history, inputs=[], outputs=[chatbot])
    
    with gr.Row():
         enable_quantization = gr.Checkbox(label="Enable 4-bit quantization", value=False)
         enable_quantization.change(reset_history, inputs=[], outputs=[chatbot])
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    # Hooking up events to callbacks
    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        generate_synthetic_data, inputs=[chatbot, model_selector, enable_quantization], outputs=[chatbot]
    )

ui.launch(inbrowser=True)